# AgentReach AI — Phase 3 Safe Test: One Google Doc only

This notebook tests read-only access to the non-sensitive Google Doc titled `AgentReach Test Document`. It searches for that exact title only. It proceeds only if exactly one native Google Doc matches, then restricts the preview tool to that file ID.

Run the cells in order. Google authorization may grant notebook code broader access than this tool uses, so inspect the code and review the Google permission prompt carefully. Running the approval cell authorizes this notebook to read only the exact-title test document for this test.

This notebook does not list unrelated files, read PDFs/Sheets, or create, edit, share, move, or delete files. If zero or multiple exact-title documents are found, the content-reading and agent tests stop.

In [1]:
%pip install -q cactus-needle google-api-python-client

## 2. Import dependencies

In [2]:
import json
import sys
import needle
from google.colab import auth
from googleapiclient.discovery import build

print('Python:', sys.version.split()[0])
print('Needle import: OK')

Python: 3.13.18
Needle import: OK


## 3. Initialize Google Drive access

If Google asks for authorization, review the requested access before approving. Do not continue if you are uncomfortable with the permission prompt.

In [3]:
auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Drive client initialized.')

Drive client initialized.


## 4. Search for the exact test title only

This query is fixed to the exact title `AgentReach Test Document`. It will not print unrelated personal document names.

In [4]:
TEST_DOC_NAME = 'AgentReach Test Document'
GOOGLE_DOCS_MIME = 'application/vnd.google-apps.document'
query = f"name = '{TEST_DOC_NAME}' and mimeType = '{GOOGLE_DOCS_MIME}' and trashed = false"
response = drive_service.files().list(
    q=query,
    pageSize=5,
    orderBy='modifiedTime desc',
    fields='files(id,name,mimeType,modifiedTime)',
).execute()
test_docs = response.get('files', [])
print('Exact-title test documents found:', len(test_docs))
if len(test_docs) == 1:
    candidate = test_docs[0]
    print('Exact test document found. Modified:', candidate.get('modifiedTime', 'unknown'))
elif not test_docs:
    print('No matching test document found. Create it first, then rerun this cell and the following cells.')
else:
    print('Multiple documents have this exact title. No document will be read; rename/remove test duplicates manually, then rerun.')

Exact-title test documents found: 1
Exact test document found. Modified: 2026-10-28T13:18:54.839Z


## 5. Select and explicitly approve one test document

Choose a number from the exact-title results above, not an ID from your other files.

In [5]:
approved_file_id = None
approved_to_read = False
if len(test_docs) == 1:
    chosen_doc = test_docs[0]
    if chosen_doc.get('name') == TEST_DOC_NAME and chosen_doc.get('mimeType') == GOOGLE_DOCS_MIME:
        approved_file_id = chosen_doc['id']
        approved_to_read = True
        print('Approved for this test only: the single exact-title Google Doc.')
        print('No other Drive files will be read by the tool.')
else:
    print('Skipped: exactly one matching test document is required. No content will be read.')

Approved for this test only: the single exact-title Google Doc.
No other Drive files will be read by the tool.


## 6. Define the restricted preview tool

This tool only accepts the approved file ID, verifies that it is a native Google Doc, and returns at most 5,000 characters.

In [6]:
@needle.tool(triggers=[r'(?i)\bpreview approved test doc\b', r'(?i)\bread the approved test document\b'])
def preview_approved_test_doc(file_id: str) -> dict:
    """Preview up to 5,000 characters from the single Google Doc titled AgentReach Test Document that the user explicitly selected and approved in this notebook. Requires its exact file ID."""
    if not approved_to_read or not approved_file_id:
        return {'ok': False, 'error': 'No test document has been explicitly approved for reading.'}
    if not isinstance(file_id, str) or file_id != approved_file_id:
        return {'ok': False, 'error': 'Access denied: only the exact user-approved test document is allowed.'}
    try:
        meta = drive_service.files().get(
            fileId=approved_file_id,
            fields='id,name,mimeType',
        ).execute()
        if meta.get('mimeType') != GOOGLE_DOCS_MIME or meta.get('name') != TEST_DOC_NAME:
            return {'ok': False, 'error': 'Only the exact-title native Google test document can be previewed.'}
        raw = drive_service.files().export(
            fileId=approved_file_id,
            mimeType='text/plain',
        ).execute()
        text = raw.decode('utf-8', errors='replace') if isinstance(raw, bytes) else str(raw)
        limit = 5000
        return {
            'ok': True,
            'name': meta.get('name'),
            'preview_characters': min(len(text), limit),
            'truncated': len(text) > limit,
            'preview': text[:limit],
            'note': 'Read-only preview of the explicitly approved test document.',
        }
    except Exception as exc:
        return {'ok': False, 'error': f'Test document preview failed: {type(exc).__name__}'}

print('Preview tool defined. Test document approved:', bool(approved_file_id))

Preview tool defined. Test document approved: True


## 7. Test the Google Docs API directly

In [7]:
if not approved_to_read or not approved_file_id:
    print('Skipped because no test document was approved.')
else:
    direct_result = preview_approved_test_doc(approved_file_id)
    print(json.dumps(direct_result, indent=2, ensure_ascii=False, default=str))
    assert direct_result.get('ok') is True, 'Direct preview failed. Inspect the safe error above.'

{
  "ok": true,
  "name": "AgentReach Test Document",
  "preview_characters": 85,
  "truncated": false,
  "preview": "﻿This is a non-sensitive test document for AgentReach AI. The integration is working.",
  "note": "Read-only preview of the explicitly approved test document."
}


## 8. Test Needle 3 tool calling

In [8]:
if not approved_to_read or not approved_file_id:
    print('Skipped because no test document was approved.')
else:
    agent = needle.Needle(tools=[preview_approved_test_doc])
    result = agent.run(
        'Preview approved test doc using preview_approved_test_doc. The approved file ID is ' + approved_file_id +
        '. Call the tool only for this exact approved test document.'
    )
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

python/cactus_needle-3.3.3-py3-none-many(…): reconstructing file:   0%|          |  0.00B /  689kB            

python/cactus_needle-3.3.3-py3-none-many(…): downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

needle3.cact: reconstructing file:   0%|          |  0.00B / 35.3MB            

needle3.cact: downloading bytes:           |  0.00B            

cactus-needle collects anonymous usage counts (function name, version, OS) to guide development; no prompts or outputs. Disable with NEEDLE_TELEMETRY=0.


{
  "type": "respond",
  "success": true,
  "error": null,
  "error_code": null,
  "reason": null,
  "function_calls": [],
  "suppressed_calls": [],
  "reasoning": "User asked for the document. Respond with the concrete details from the result.",
  "confidence": 0.4846,
  "prefill_tps": 79.9,
  "decode_tps": 73.3,
  "peak_ram_mb": 238.2,
  "results": [
    {
      "ok": true,
      "name": "AgentReach Test Document",
      "preview_characters": 85,
      "truncated": false,
      "preview": "﻿This is a non-sensitive test document for AgentReach AI. The integration is working.",
      "note": "Read-only preview of the explicitly approved test document."
    }
  ]
}


## Success checkpoint

- The search finds exactly one native Google Doc with the exact title `AgentReach Test Document`.
- The direct API preview returns `ok: true` and the harmless sample text.
- The Needle result includes the tool execution result for that same document.

No interactive input prompts are used, so you do not need to type confirmation phrases into a Python code cell. If zero or multiple exact-title documents are found, the notebook stops before reading content. If the direct preview succeeds but Needle does not call the tool, Google Docs access works and the remaining issue is tool selection. No writes or deletes are included.